# NB08: Summary Visualizations

**Purpose**: Generate publication-quality figures summarizing the Rosetta mapping.

**Figures**:
1. Confidence tier distribution (bar chart)
2. UpSet plot of tier intersections
3. Per-channel reaction coverage (horizontal bar)
4. Validation F1 per channel (grouped bar: precision, recall, F1)
5. Coverage vs number of supporting channels (histogram)

**Output**: PNG files in `../figures/`

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from upsetplot import UpSet
import gc

DATA_DIR = '../data'
FIG_DIR = '../figures'

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
})

df = pd.read_parquet(f'{DATA_DIR}/evidence_integration_summary.parquet')
print(f'Loaded {len(df):,} reactions ({df["any_evidence"].sum():,} with evidence)')

Loaded 34,343 reactions (17,351 with evidence)


## Figure 1: Confidence Tier Distribution

In [2]:
tier_order = ['high', 'medium', 'low', 'none']
tier_colors = {'high': '#2ca02c', 'medium': '#ff7f0e', 'low': '#d62728', 'none': '#bdbdbd'}
counts = df['confidence'].value_counts().reindex(tier_order)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(tier_order, counts, color=[tier_colors[t] for t in tier_order], edgecolor='white', linewidth=0.5)
for bar, ct in zip(bars, counts):
    pct = 100 * ct / len(df)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{ct:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Balanced reactions')
ax.set_xlabel('Confidence tier')
ax.set_title('Rosetta Mapping: Confidence Distribution\n(34,343 mass-balanced ModelSEED reactions)')
ax.set_ylim(0, max(counts) * 1.2)
ax.spines[['top', 'right']].set_visible(False)

fig.savefig(f'{FIG_DIR}/confidence_distribution.png')
plt.show()
print('Saved: confidence_distribution.png')

Saved: confidence_distribution.png


## Figure 2: UpSet Plot of Tier Intersections

In [3]:
upset_df = df.set_index(
    pd.MultiIndex.from_frame(
        df[['has_tier1', 'has_tier2', 'has_tier3', 'has_rast']].rename(
            columns={'has_tier1': 'Tier 1\n(UniProt)', 'has_tier2': 'Tier 2\n(Pangenome)',
                     'has_tier3': 'Tier 3\n(Curated)', 'has_rast': 'RAST'}
        )
    )
)

upset = UpSet(upset_df, subset_size='count', show_counts=True,
              sort_by='cardinality', min_subset_size=50)
fig = plt.figure(figsize=(12, 6))
upset.plot(fig=fig)
fig.suptitle('Evidence Tier Intersections Across 34,343 Balanced Reactions', y=1.02, fontsize=13)
fig.savefig(f'{FIG_DIR}/tier_upset.png')
plt.show()
print('Saved: tier_upset.png')
del upset_df

Saved: tier_upset.png


/home/seaver/.local/lib/python3.13/site-packages/upsetplot/plotting.py:795: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  styles["linewidth"].fillna(1, inplace=True)
/home/seaver/.local/lib/python3.13/site-packages/upsetplot/plotting.py:796: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

## Figure 3: Per-Channel Reaction Coverage

In [4]:
channel_cols = ['T1_ec', 'T1_brenda', 'T1_rhea', 'T2_eggnog_ec', 'T2_bakta_ec',
                'T3_paperblast', 'T3_seedclass', 'T3_besthitmetacyc_ec',
                'T3_besthitmetacyc_rxnid', 'RAST']
labels = ['UniProt EC', 'BRENDA', 'Rhea', 'eggNOG EC', 'bakta EC',
          'PaperBLAST', 'seedclass', 'MetaCyc (EC)', 'MetaCyc (rxnId)', 'RAST']
colors_ch = ['#1f77b4', '#1f77b4', '#1f77b4', '#ff7f0e', '#ff7f0e',
             '#2ca02c', '#2ca02c', '#2ca02c', '#2ca02c', '#9467bd']

coverage = [df[c].sum() for c in channel_cols]
pcts = [100 * c / len(df) for c in coverage]

fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(labels))
bars = ax.barh(y_pos, pcts, color=colors_ch, edgecolor='white', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel('Reaction coverage (%)')
ax.set_title('Per-Channel Coverage of Balanced Reactions')
ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)

for bar, ct, pct in zip(bars, coverage, pcts):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{ct:,} ({pct:.1f}%)', va='center', fontsize=9)

ax.legend(handles=[
    plt.Rectangle((0,0),1,1, fc='#1f77b4', label='Tier 1 (UniProt-native)'),
    plt.Rectangle((0,0),1,1, fc='#ff7f0e', label='Tier 2 (Pangenome)'),
    plt.Rectangle((0,0),1,1, fc='#2ca02c', label='Tier 3 (Curated)'),
    plt.Rectangle((0,0),1,1, fc='#9467bd', label='RAST (validation)'),
], loc='lower right', fontsize=9)

fig.savefig(f'{FIG_DIR}/channel_coverage.png')
plt.show()
print('Saved: channel_coverage.png')

Saved: channel_coverage.png


## Figure 4: RAST Validation — EC-Level F1 per Channel

In [5]:
ec_bridge = pd.read_parquet(f'{DATA_DIR}/ec_to_reaction.parquet')
bridge_ecs = set(ec_bridge['ec'])
rast_data = pd.read_parquet(f'{DATA_DIR}/rast_protein_ec.parquet', columns=['ec'])
rast_ecs_set = set(rast_data['ec'])
del rast_data
gc.collect()

def load_tier_ecs(path, channel_col, channel_vals):
    d = pd.read_parquet(path, columns=[channel_col, 'ec'])
    result = {}
    for val, label in channel_vals:
        if channel_col == 'channels':
            result[label] = set(d[d[channel_col].str.contains(val, na=False)]['ec'])
        else:
            result[label] = set(d[d[channel_col] == val]['ec'])
    del d
    gc.collect()
    return result

t1_ecs = load_tier_ecs(f'{DATA_DIR}/uniprot_native_protein_ec.parquet', 'channels',
                       [('uniprot_ec', 'UniProt EC'), ('brenda', 'BRENDA'), ('rhea', 'Rhea')])
t2_ecs = load_tier_ecs(f'{DATA_DIR}/pangenome_gc_ec.parquet', 'channels',
                       [('eggnog_ec', 'eggNOG EC'), ('bakta_ec', 'bakta EC')])
t3_ecs = load_tier_ecs(f'{DATA_DIR}/curated_evidence_ec.parquet', 'channel',
                       [('paperblast', 'PaperBLAST'), ('seedclass', 'seedclass'),
                        ('besthitmetacyc_ec', 'MetaCyc')])

all_channel_ecs = {**t1_ecs, **t2_ecs, **t3_ecs}

val_rows = []
for name, ec_set in all_channel_ecs.items():
    tp = len(ec_set & rast_ecs_set)
    prec = tp / len(ec_set) if ec_set else 0
    recall = tp / len(rast_ecs_set) if rast_ecs_set else 0
    f1 = 2 * prec * recall / (prec + recall) if (prec + recall) > 0 else 0
    val_rows.append({'channel': name, 'precision': prec, 'recall': recall, 'f1': f1})

val_df = pd.DataFrame(val_rows)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(val_df))
w = 0.25
ax.bar(x - w, val_df['precision'], w, label='Precision', color='#1f77b4')
ax.bar(x, val_df['recall'], w, label='Recall', color='#ff7f0e')
ax.bar(x + w, val_df['f1'], w, label='F1', color='#2ca02c')
ax.set_xticks(x)
ax.set_xticklabels(val_df['channel'], rotation=35, ha='right')
ax.set_ylabel('Score')
ax.set_title('EC-Level Validation Against RAST')
ax.set_ylim(0, 1.05)
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

fig.savefig(f'{FIG_DIR}/ec_validation_f1.png')
plt.show()
print('Saved: ec_validation_f1.png')
del ec_bridge, all_channel_ecs, t1_ecs, t2_ecs, t3_ecs
gc.collect()

Saved: ec_validation_f1.png


0

## Figure 5: Reactions by Number of Supporting Channels

In [6]:
chan_counts = df['n_channels'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#bdbdbd'] + ['#2ca02c' if n >= 3 else '#ff7f0e' if n >= 1 else '#bdbdbd'
                         for n in range(1, len(chan_counts))]
ax.bar(chan_counts.index, chan_counts.values, color=colors, edgecolor='white', linewidth=0.5)

for i, (n, ct) in enumerate(chan_counts.items()):
    if ct > 200:
        ax.text(n, ct + 150, f'{ct:,}', ha='center', fontsize=9)

ax.set_xlabel('Number of supporting evidence channels')
ax.set_ylabel('Reactions')
ax.set_title('Evidence Depth per Reaction')
ax.spines[['top', 'right']].set_visible(False)

n_mapped = int(df['any_evidence'].sum())
n_multi = int((df['n_channels'] >= 2).sum())
ax.annotate(f'{n_mapped:,} mapped ({100*n_mapped/len(df):.1f}%)\n'
            f'{n_multi:,} with 2+ channels',
            xy=(7, chan_counts.get(7, 0)), xytext=(8.5, max(chan_counts)*0.7),
            fontsize=10, ha='center',
            bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='gray'))

fig.savefig(f'{FIG_DIR}/evidence_depth.png')
plt.show()
print('Saved: evidence_depth.png')

Saved: evidence_depth.png


## Summary Statistics

In [7]:
print('=' * 60)
print('ROSETTA MAPPING SUMMARY')
print('=' * 60)

total = len(df)
mapped = int(df['any_evidence'].sum())
conf = df['confidence'].value_counts()

print(f'\nTotal balanced reactions: {total:,}')
print(f'Mapped (any evidence):   {mapped:,} ({100*mapped/total:.1f}%)')
print(f'  High confidence:       {conf.get("high", 0):,} ({100*conf.get("high", 0)/total:.1f}%)')
print(f'  Medium confidence:     {conf.get("medium", 0):,} ({100*conf.get("medium", 0)/total:.1f}%)')
print(f'  Low confidence:        {conf.get("low", 0):,} ({100*conf.get("low", 0)/total:.1f}%)')
print(f'  No evidence:           {conf.get("none", 0):,} ({100*conf.get("none", 0)/total:.1f}%)')

print(f'\nTier coverage:')
for col, label in [('has_tier1', 'Tier 1 (UniProt-native)'),
                    ('has_tier2', 'Tier 2 (Pangenome)'),
                    ('has_tier3', 'Tier 3 (Curated)'),
                    ('has_rast', 'RAST (validation)')]:
    n = int(df[col].sum())
    print(f'  {label:<30s} {n:>6,} ({100*n/total:.1f}%)')

print(f'\nMulti-evidence support:')
for thresh in [2, 3, 5]:
    n = int((df['n_channels'] >= thresh).sum())
    print(f'  {thresh}+ channels: {n:,} ({100*n/total:.1f}%)')

print(f'\nProtein-level F1 (Tier 1 vs RAST): 0.8372')
print(f'  Precision: 0.8473, Recall: 0.8273')
print(f'\nH0 rejected: YES (coverage >50%, F1 >0.7)')
print(f'H1 supported: NO (coverage 50.4%, below 70% threshold)')

print(f'\nFigures saved to: {FIG_DIR}/')
print(f'\nNext: /synthesize to write REPORT.md')

ROSETTA MAPPING SUMMARY

Total balanced reactions: 34,343
Mapped (any evidence):   17,351 (50.5%)
  High confidence:       14,470 (42.1%)
  Medium confidence:     2,752 (8.0%)
  Low confidence:        129 (0.4%)
  No evidence:           16,992 (49.5%)

Tier coverage:
  Tier 1 (UniProt-native)        17,215 (50.1%)
  Tier 2 (Pangenome)             14,557 (42.4%)
  Tier 3 (Curated)               16,290 (47.4%)
  RAST (validation)              10,730 (31.2%)

Multi-evidence support:
  2+ channels: 16,491 (48.0%)
  3+ channels: 16,031 (46.7%)
  5+ channels: 13,266 (38.6%)

Protein-level F1 (Tier 1 vs RAST): 0.8372
  Precision: 0.8473, Recall: 0.8273

H0 rejected: YES (coverage >50%, F1 >0.7)
H1 supported: NO (coverage 50.4%, below 70% threshold)

Figures saved to: ../figures/

Next: /synthesize to write REPORT.md
